In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
import os
from src.preprocessing import ensure_directories, initial_cleaning, normalize_text, process_and_tokenize, print_reduction_rates, parallel_process, wrapper_normalize, wrapper_tokenize

ensure_directories()

file_path = "../data/raw/995,000_rows.csv"

if os.path.exists(file_path):
    df_sample = pd.read_csv(file_path, low_memory=False, dtype={'Unnamed: 0': str, 'id': str})
else:
    print("File not found")

print(f"DataFrame has {df_sample.shape[0]} rows and {df_sample.shape[1]} columns")

print("--- Columns and DTypes ---")
print(df_sample.dtypes)

print("\n--- Missing values per column ---")
print(df_sample.isnull().sum())

df_sample.head()

DataFrame has 995000 rows and 17 columns
--- Columns and DTypes ---
Unnamed: 0              str
id                      str
domain                  str
type                    str
url                     str
content                 str
scraped_at              str
inserted_at             str
updated_at              str
title                   str
authors                 str
keywords            float64
meta_keywords           str
meta_description        str
tags                    str
summary             float64
source                  str
dtype: object

--- Missing values per column ---
Unnamed: 0               1
id                       7
domain                  11
type                 47786
url                     11
content                 12
scraped_at              13
inserted_at             13
updated_at              13
title                 8606
authors             442757
keywords            995000
meta_keywords        38790
meta_description    525106
tags                764081
su

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary,source
0,732,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,Plus one article on Google Plus\n\n(Thanks to ...,2017-11-27T01:14:42.983556,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,Iran News Round Up,NaN,NaN,"['National Review', 'National Review Online', ...",NaN,NaN,NaN,NaN
1,1348,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,The Cost Of The Best Senate Banking Committee ...,2017-11-27T01:14:08.7454,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,The Cost Of The Best Senate Banking Committee ...,NaN,NaN,[''],NaN,NaN,NaN,NaN
2,7119,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,Man Awoken From 27-Year Coma Commits Suicide A...,2017-11-27T01:14:21.395055,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,Man Awoken From 27-Year Coma Commits Suicide A...,NaN,NaN,[''],NaN,NaN,NaN,NaN
3,1518,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,WHEN Julia Geist was asked to draw a picture o...,2018-02-11 00:46:42.632962,2018-02-11 00:14:20.346838,2018-02-11 00:14:20.346871,Opening a Gateway for Girls to Enter the Compu...,NaN,NaN,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,NaN,NaN,nytimes
4,9345,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,– 100 Compiled Studies on Vaccine Dangers (Act...,2017-11-10T11:18:44.524042,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,100 Compiled Studies on Vaccine Dangers – Infi...,NaN,NaN,[''],NaN,"Lymphoma, Hepatitis B, Immune System, Health, ...",NaN,NaN


In [4]:
df_cleaned = initial_cleaning(df_sample)
print(df_cleaned.head())
print("\n--- Missing values per column ---")
print(df_cleaned.isnull().sum())

Cleaning done: Start=995000, End=947213, Dropped=47787 rows.
          id               domain        type  \
0  7444726.0   nationalreview.com   political   
1  6213642.0    beforeitsnews.com        fake   
2  3867639.0     dailycurrant.com      satire   
3  9560791.0          nytimes.com    reliable   
4  2059625.0  infiniteunknown.net  conspiracy   

                                                 url  \
0  http://www.nationalreview.com/node/152734/%E2%...   
1  http://beforeitsnews.com/economy/2012/06/the-c...   
2  http://dailycurrant.com/2016/01/18/man-awoken-...   
3  https://query.nytimes.com/gst/fullpage.html?re...   
4  http://www.infiniteunknown.net/2011/09/14/100-...   

                                             content  \
0  Plus one article on Google Plus\n\n(Thanks to ...   
1  The Cost Of The Best Senate Banking Committee ...   
2  Man Awoken From 27-Year Coma Commits Suicide A...   
3  WHEN Julia Geist was asked to draw a picture o...   
4  – 100 Compiled Studies o

In [5]:
test_article = """
    BREAKING NEWS!!! Check out https://fakenews.com/article.
    Contact us at spam@email.com.
    This happened on 2023-10-25 and cost 1000000 dollars.
"""

print(normalize_text(test_article))

breaking news check out https fakenewscom article contact us at emailtoken this happened on datetoken and cost numtoken dollars


In [6]:
print("starting normalization of 'content' and 'title' columns using parallel processing")
df_cleaned = parallel_process(df_cleaned, wrapper_normalize)

df_cleaned[['title', 'content']].head()

starting normalization of 'content' and 'title' columns using parallel processing


,title,content
0,iran news round up,plus one article on google plus thanks to ali ...
1,the cost of the best senate banking committee ...,the cost of the best senate banking committee ...
2,man awoken from numtoken year coma commits sui...,man awoken from numtoken year coma commits sui...
3,opening a gateway for girls to enter the compu...,when julia geist was asked to draw a picture o...
4,numtoken compiled studies on vaccine dangers i...,numtoken compiled studies on vaccine dangers a...


In [7]:
import random

# Vælg 3 tilfældige indexer fra vores rensede dataframe
random_indices = random.sample(range(len(df_cleaned)), 3)

for idx in random_indices:
    print(f"--- Artikel {idx} ---")
    # Vi printer kun de første 500 tegn for overskuelighedens skyld
    print(df_cleaned['content'].iloc[idx][:500])
    print("\n")

--- Artikel 221501 ---
the somali peace talks under way in kenyas capital are in danger of collapsing unless stronger leadership is provided by the mediators somali participants and the international community a brussels based policy group warned without new energy and focus the peace talks will likely fissure along all too predictable lines the international crisis group said in a report marc lacey nyt


--- Artikel 486076 ---
michel barnier insisted the remaining eu numtoken states would not pay for the burden of britains decision to leave the bloc after the latest round of exit negotiations speaking after the fourth round of talks mr davis reiterated in his own address to the media that the uk was making decisive steps forward a body language expert professor alain laurent verbeke claimed the eus chief negotiator was more credible in brexit negotiations so far looking at an image of the first round of talks he told 


--- Artikel 138213 ---
daily kos world of warcraft update a simpl

In [8]:
print("Running NLTK Tokenization, Stopwords and Stemming using parallel processing")
df_cleaned = parallel_process(df_cleaned, wrapper_tokenize)
df_cleaned.head()

Running NLTK Tokenization, Stopwords and Stemming using parallel processing


,id,domain,type,url,content,scraped_at,title,authors,meta_keywords,meta_description,tags,source
0,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,plus one articl googl plus thank ali alfoneh a...,2017-11-27 01:14:42.983556,iran news round up,unknown_author,"['National Review', 'National Review Online', ...",NaN,NaN,NaN
1,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,cost best senat bank committe jp morgan buy $ ...,2017-11-27 01:14:08.745400,the cost of the best senate banking committee ...,unknown_author,[''],NaN,NaN,NaN
2,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,man awoken numtoken year coma commit suicid le...,2017-11-27 01:14:21.395055,man awoken from numtoken year coma commits sui...,unknown_author,[''],NaN,NaN,NaN
3,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,julia geist ask draw pictur comput scientist l...,NaT,opening a gateway for girls to enter the compu...,unknown_author,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,NaN,nytimes
4,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,numtoken compil studi vaccin danger activist p...,2017-11-10 11:18:44.524042,numtoken compiled studies on vaccine dangers i...,unknown_author,[''],NaN,"Lymphoma, Hepatitis B, Immune System, Health, ...",NaN


In [9]:
output_path = '../data/processed/995K_cleaned.csv'
df_cleaned.to_csv(output_path, index=False)

In [10]:
# Calculate reductionrates
print_reduction_rates()


--- Final reduction rates ---
Original vocabulary: 1,737,026 unique tokens
After stopwords: 1,736,875 tokens (0.01% reduction)
After stemming: 1,496,298 tokens (13.85% reduction from previous)
Total reduction: 13.86%
